# Question B1 (15 marks)

Real world datasets often have a mix of numeric and categorical features – this dataset is one example. To build models on such data, categorical features have to be encoded or embedded.

PyTorch Tabular is a library that makes it very convenient to build neural networks for tabular data. It is built on top of PyTorch Lightning, which abstracts away boilerplate model training code and makes it easy to integrate other tools, e.g. TensorBoard for experiment tracking.

For questions B1 and B2, the following features should be used:   
- **Numeric / Continuous** features: dist_to_nearest_stn, dist_to_dhoby, degree_centrality, eigenvector_centrality, remaining_lease_years, floor_area_sqm
- **Categorical** features: month, town, flat_model_type, storey_range



---



In [65]:
# !pip install pytorch_tabular[extra]

In [66]:
# !pip install torch_optimizer

In [67]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from pytorch_tabular import TabularModel
from pytorch_tabular.models import CategoryEmbeddingModelConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

from sklearn.metrics import mean_squared_error, r2_score

## Configuration for Google Colab
import torch
import omegaconf
import typing
import collections

torch.serialization.add_safe_globals([omegaconf.dictconfig.DictConfig])
torch.serialization.add_safe_globals([omegaconf.base.ContainerMetadata])
torch.serialization.add_safe_globals([typing.Any])
torch.serialization.add_safe_globals([dict])
torch.serialization.add_safe_globals([collections.defaultdict])
torch.serialization.add_safe_globals([omegaconf.listconfig.ListConfig])
torch.serialization.add_safe_globals([list])
torch.serialization.add_safe_globals([int])
torch.serialization.add_safe_globals([omegaconf.nodes.AnyNode])
torch.serialization.add_safe_globals([omegaconf.base.Metadata])

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/SC4001 Neural Network')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1.Divide the dataset (‘hdb_price_prediction.csv’) into train, validation and test sets by using entries from year 2019 and before as training data, year 2020 as validation data and year 2021 as test data.
**Do not** use data from year 2022 and year 2023.



In [68]:
df = pd.read_csv('/content/drive/MyDrive/SC4001 Neural Network/hdb_price_prediction.csv')

# YOUR CODE HERE

# Filter Out data from year 2022 and 2023
df = df[df['year'].isin([2019, 2020, 2021]) | (df['year'] < 2019)]

# Define features that should be used
numeric_features     = ["dist_to_nearest_stn", "dist_to_dhoby", "degree_centrality", "eigenvector_centrality", "remaining_lease_years", "floor_area_sqm"]
categorical_features = ["month", "town", "flat_model_type", "storey_range"]
prediction_target    = ["resale_price"]
all_features         = numeric_features + categorical_features + prediction_target

# Split features for train, validation & test by year
train_df = df[df['year'] <= 2019]
val_df   = df[df['year'] == 2020]
test_df  = df[df['year'] == 2021]

# Separate features and target
train_df = train_df[all_features]
val_df   = val_df[all_features]
test_df  = test_df[all_features]

In [69]:
print("Train Data Shape:", train_df.shape)
print("Val Data Shape:", val_df.shape)
print("Test Data Shape:", test_df.shape)

Train Data Shape: (64057, 11)
Val Data Shape: (23313, 11)
Test Data Shape: (29057, 11)


In [70]:
# Make sure data sets have no null values
print(train_df.isnull().sum().sum())
print(val_df.isnull().sum().sum())
print(test_df.isnull().sum().sum())

0
0
0


2.Refer to the documentation of **PyTorch Tabular** and perform the following tasks: https://pytorch-tabular.readthedocs.io/en/latest/#usage
- Use **[DataConfig](https://pytorch-tabular.readthedocs.io/en/latest/data/)** to define the target variable, as well as the names of the continuous and categorical variables.
- Use **[TrainerConfig](https://pytorch-tabular.readthedocs.io/en/latest/training/)** to automatically tune the learning rate. Set batch_size to be 1024 and set max_epoch as 50.
- Use **[CategoryEmbeddingModelConfig](https://pytorch-tabular.readthedocs.io/en/latest/models/#category-embedding-model)** to create a feedforward neural network with 1 hidden layer containing 50 neurons.
- Use **[OptimizerConfig](https://pytorch-tabular.readthedocs.io/en/latest/optimizer/)** to choose Adam optimiser. There is no need to set the learning rate (since it will be tuned automatically) nor scheduler.
- Use **[TabularModel](https://pytorch-tabular.readthedocs.io/en/latest/tabular_model/)** to initialise the model and put all the configs together.

In [71]:
batch_size = 1024
max_epochs = 50
layers = "50"

In [72]:
# YOUR CODE HERE

data_config = DataConfig(
    target = prediction_target,
    continuous_cols = numeric_features,
    categorical_cols = categorical_features
)

trainer_config_with_auto_lr = TrainerConfig(
    auto_lr_find = True,
    batch_size = batch_size,
    max_epochs = max_epochs
)

model_config = CategoryEmbeddingModelConfig(
    task = "regression",
    layers = layers
)

optimizer_config = OptimizerConfig()

tabular_model_with_auto_lr = TabularModel(
    data_config = data_config,
    model_config = model_config,
    optimizer_config = optimizer_config,
    trainer_config = trainer_config_with_auto_lr
)

2026-03-14 05:23:56,915 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


In [73]:
tabular_model_with_auto_lr.fit(train = train_df, validation = val_df)
suggested_lr = tabular_model_with_auto_lr.trainer.lightning_module.hparams.get('learning_rate')
print(f"\nSuggested LR from auto_lr_find: {suggested_lr}")

INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-03-14 05:23:56,953 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-14 05:23:57,026 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-14 05:23:57,269 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: CategoryEmbeddingModel
2026-03-14 05:23:57,333 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-14 05:23:57,405 - {pytorch_tabular.tabular_model:655} - INFO - Auto LR Find Sta

Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/.lr_find_d3b04b85-55cd-45bc-9e97-edf232537c4a.ckpt
INFO:pytorch_lightning.utilities.rank_zero:Restored all states from the checkpoint at /content/.lr_find_d3b04b85-55cd-45bc-9e97-edf232537c4a.ckpt
INFO:pytorch_lightning.tuner.lr_finder:Learning rate set to 2.7542287033381663e-05
2026-03-14 05:24:01,309 - {pytorch_tabular.tabular_model:668} - INFO - Suggested LR: 2.7542287033381663e-05. For plot and detailed analysis, use `find_learning_rate` method.
2026-03-14 05:24:01,311 - {pytorch_tabular.tabular_model:677} - INFO - Training Started


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │  2.9 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer          │  1.5 K │ train │     0 │
│ 2 │ head             │ LinearHead                │     51 │ train │     0 │
│ 3 │ loss             │ MSELoss                   │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 4.5 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.5 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


2026-03-14 05:27:10,953 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-14 05:27:10,954 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model



Suggested LR from auto_lr_find: 2.7542287033381663e-05


In [74]:
pred_df_with_auto_lr = tabular_model_with_auto_lr.predict(test_df)

test_values_with_auto_lr  = test_df["resale_price"].values
pred_values_with_auto_lr  = pred_df_with_auto_lr["resale_price_prediction"].values

# Calculate MSE and R2
test_mse_with_auto_lr = mean_squared_error(test_values_with_auto_lr, pred_values_with_auto_lr)
test_r2_with_auto_lr  = r2_score(test_values_with_auto_lr, pred_values_with_auto_lr)

print(f"With Auto LR Test MSE: {test_mse_with_auto_lr:.4f}")
print(f"With Auto LR Test R2:  {test_r2_with_auto_lr:.4f}")

With Auto LR Test MSE: 287991537572.3720
With Auto LR Test R2:  -9.8873


When auto_lr_find is set to True, the learning rate found was ridiculously miniscule (2.7542287033381663e-05), such that it the model is not able to learn anything from the dataset during training. As a result, we achieved a negative R² of -9.8873 and an extremely large MSE of 287991537572.3720. The same code is tried running in different devices, different laptops, using google Colab in different accounts, using Jupyter Notebook, everything led to the same negative R^2 result when auto_lr_find is used.

Thus, I have decided to use my own learning_rate below instead of using auto_lr_find. I've set it to be 0.01.

In [75]:
trainer_config = TrainerConfig(
    auto_lr_find = False,
    batch_size = batch_size,
    max_epochs = max_epochs
)

tabular_model = TabularModel(
    data_config = data_config,
    model_config = model_config,
    optimizer_config = optimizer_config,
    trainer_config = trainer_config
)

tabular_model.config['learning_rate'] = 0.01

2026-03-14 05:27:11,902 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


In [76]:
# Train the model
tabular_model.fit(train = train_df, validation = val_df)

INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-03-14 05:27:11,933 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-14 05:27:12,002 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-14 05:27:12,238 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: CategoryEmbeddingModel
2026-03-14 05:27:12,309 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-14 05:27:12,382 - {pytorch_tabular.tabular_model:677} - INFO - Training Started

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │  2.9 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer          │  1.5 K │ train │     0 │
│ 2 │ head             │ LinearHead                │     51 │ train │     0 │
│ 3 │ loss             │ MSELoss                   │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 4.5 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.5 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is 
set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


2026-03-14 05:30:17,427 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-14 05:30:17,427 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


3.Report the test MSE error and the test R2 value that you obtained.



In [77]:
# YOUR CODE & RESULT HERE
evaluation = tabular_model.evaluate(test_df)
pred_df = tabular_model.predict(test_df)

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │       6372398080.0        │
│  test_mean_squared_error  │       6372398080.0        │
└───────────────────────────┴───────────────────────────┘

In [78]:
test_values  = test_df["resale_price"].values
pred_values  = pred_df["resale_price_prediction"].values

# Calculate MSE and R2
test_mse = mean_squared_error(test_values, pred_values)
test_r2  = r2_score(test_values, pred_values)

print(f"Test MSE: {test_mse:.4f}")
print(f"Test R2:  {test_r2:.4f}")

Test MSE: 6372397896.7095
Test R2:  0.7591


4.Print out the corresponding rows in the dataframe for the top 20 test samples with the largest errors. Are there any patterns or common characteristics among these data points? Based on your observations, suggest and briefly explain a potential strategy to improve the model on these cases.



In [79]:
# YOUR CODE & RESULT HERE

errors = np.abs(test_values - pred_values)

# Get top 20 largest errors
top20_indices = np.argsort(errors)[-20:][::-1]
top20_df = test_df.iloc[top20_indices].copy()
top20_df['predicted_price'] = pred_values[top20_indices]
top20_df['absolute_error'] = errors[top20_indices]

print("Top 20 Test Samples with the Largest Errors:")
display(top20_df.sort_values('absolute_error', ascending = False))

Top 20 Test Samples with the Largest Errors:


,dist_to_nearest_stn,dist_to_dhoby,degree_centrality,eigenvector_centrality,remaining_lease_years,floor_area_sqm,month,town,flat_model_type,storey_range,resale_price,predicted_price,absolute_error
92405,0.581977,2.309477,0.016807,0.047782,50.166667,88.0,11,BUKIT MERAH,"3 ROOM, Standard",01 TO 03,780000.0,3.496576e+05,430342.40625
106192,0.658035,3.807573,0.016807,0.008342,93.333333,109.0,12,QUEENSTOWN,"4 ROOM, Premium Apartment Loft",04 TO 06,968000.0,5.624829e+05,405517.12500
105372,0.570988,4.922054,0.016807,0.005350,46.916667,134.0,2,QUEENSTOWN,"4 ROOM, Terrace",01 TO 03,975000.0,5.831164e+05,391883.56250
105695,0.745596,3.720593,0.016807,0.008342,93.916667,97.0,6,QUEENSTOWN,"4 ROOM, Premium Apartment Loft",07 TO 09,930000.0,5.463939e+05,383606.12500
106057,0.584731,3.882019,0.016807,0.008342,93.500000,97.0,10,QUEENSTOWN,"4 ROOM, Premium Apartment Loft",13 TO 15,958000.0,5.766996e+05,381300.43750
105699,0.745596,3.720593,0.016807,0.008342,93.916667,109.0,6,QUEENSTOWN,"4 ROOM, Premium Apartment Loft",31 TO 33,1032888.0,6.517622e+05,381125.75000
105869,0.554599,4.841933,0.016807,0.008342,46.416667,120.0,8,QUEENSTOWN,"4 ROOM, Terrace",01 TO 03,930000.0,5.525546e+05,377445.37500
90957,1.292540,10.763777,0.016807,0.000217,75.583333,144.0,6,BUKIT BATOK,"EXECUTIVE, Apartment",10 TO 12,968000.0,5.991620e+05,368838.00000
105468,0.701852,3.763948,0.016807,0.008342,94.166667,95.0,3,QUEENSTOWN,"4 ROOM, Premium Apartment Loft",10 TO 12,920000.0,5.560328e+05,363967.25000
105696,0.658035,3.807573,0.016807,0.008342,93.916667,109.0,6,QUEENSTOWN,"4 ROOM, Premium Apartment Loft",10 TO 12,950000.0,5.897920e+05,360208.00000


In [80]:
def display_with_title(title, df):
    display(df.style.set_caption(title).set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('text-align', 'left'), ('padding', '10px 0px')]}
    ]))

display_with_title(
    "Town Distribution",
    top20_df['town'].value_counts().reset_index().rename(columns = {'town': 'Town', 'count': 'Count'})
)

display_with_title(
    "Flat Model Type Distribution",
    top20_df['flat_model_type'].value_counts().reset_index().rename(columns = {'flat_model_type': 'Flat Model Type', 'count': 'Count'})
)

display_with_title(
    "Storey Range Distribution",
    top20_df['storey_range'].value_counts().reset_index().rename(columns = {'storey_range': 'Storey Range', 'count': 'Count'})
)

display_with_title(
    "Numeric Feature Statistics",
    top20_df[numeric_features + ['resale_price', 'predicted_price', 'absolute_error']].describe().round(2)
)

display_with_title(
    "Resale Price Comparison",
    pd.DataFrame({
        'Metric': ['Min Price', 'Max Price', 'Mean Price', 'Overall Test Mean', 'Top 20 Error Mean'],
        'Value': [
            f"${top20_df['resale_price'].min():,.0f}",
            f"${top20_df['resale_price'].max():,.0f}",
            f"${top20_df['resale_price'].mean():,.0f}",
            f"${test_df['resale_price'].mean():,.0f}",
            f"${top20_df['resale_price'].mean():,.0f}"
        ]
    })
)

,Town,Count
0,QUEENSTOWN,14
1,BUKIT MERAH,2
2,BUKIT BATOK,1
3,KALLANG/WHAMPOA,1
4,WOODLANDS,1
5,BISHAN,1


,Flat Model Type,Count
0,"4 ROOM, Premium Apartment Loft",12
1,"3 ROOM, Standard",2
2,"4 ROOM, Terrace",2
3,"EXECUTIVE, Apartment",2
4,"3 ROOM, Terrace",1
5,"5 ROOM, DBSS",1


,Storey Range,Count
0,01 TO 03,5
1,10 TO 12,4
2,04 TO 06,2
3,31 TO 33,2
4,07 TO 09,1
5,13 TO 15,1
6,25 TO 27,1
7,34 TO 36,1
8,19 TO 21,1
9,16 TO 18,1


,dist_to_nearest_stn,dist_to_dhoby,degree_centrality,eigenvector_centrality,remaining_lease_years,floor_area_sqm,resale_price,predicted_price,absolute_error
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.710000,4.840000,0.020000,0.010000,80.200000,114.550000,970879.100000,605525.880000,365353.200000
std,0.180000,3.350000,0.000000,0.020000,19.590000,32.690000,140955.540000,148755.950000,24724.670000
min,0.420000,2.060000,0.020000,0.000000,46.420000,88.000000,680888.000000,342550.620000,336991.750000
25%,0.580000,3.720000,0.020000,0.010000,66.080000,97.000000,930000.000000,557017.590000,346025.530000
50%,0.700000,3.790000,0.020000,0.010000,93.380000,97.000000,963000.000000,591479.250000,359540.000000
75%,0.750000,4.120000,0.020000,0.010000,93.710000,120.000000,976250.000000,628445.640000,381169.420000
max,1.290000,16.950000,0.030000,0.050000,94.170000,210.000000,1360000.000000,1023008.250000,430342.410000


,Metric,Value
0,Min Price,"$680,888"
1,Max Price,"$1,360,000"
2,Mean Price,"$970,879"
3,Overall Test Mean,"$511,417"
4,Top 20 Error Mean,"$970,879"


Observations:
The top 20 largest errors are concentrated in high-value properties with a mean resale price of 970,879 dollars, which is nearly double the overall test mean of 511,417 dollars. Queenstown dominates with 14 out of 20 samples, and 4-Room Premium Apartment Loft accounts for 12 out of 20 samples, suggesting that the model systematically underpredicts premium flat types in high-value mature estates.

How to Improve:
These high-value properties are very likely underrepresented in the training data, causing the model to be biased towards predicting average prices. A potential strategy is to add more representative features such as average price per town, which would give the model better signals to differentiate premium properties. Additionally, increasing the model complexity by adding more hidden layers or neurons could also improve the model's capacity to learn the non-linear pricing patterns of these rare premium flat types.

In [81]:
tabular_model.save_model('/content/drive/MyDrive/SC4001 Neural Network/b1model')

2026-03-14 05:30:18,997 - {pytorch_tabular.tabular_model:1577} - WARNING - Directory is not empty. Overwriting the contents.
INFO:pytorch_lightning.trainer.connectors.checkpoint_connector:`weights_only` was not set, defaulting to `False`.
